# N-Body Simulation

This simulation models the motion of $N$ masses.

We demonstrate this with random setups, a Sun-Earth-Moon setup, and a galaxy setup which utilises the improved performance of the Barnes-Hut tree solver.  

### TODO
- Implement Barnes-Hut tree solver

## Load libraries

In [ ]:
# Numerical calculations
import numpy as np

# ODE Solving
from scipy.integrate import solve_ivp

# Graphics / plotting
import matplotlib.pyplot as plt

## Parameters

In [ ]:
G = 6.67e-11
G = 1

## Gravitational Force between Two Objects

In [ ]:
# Force between two objects
def compute_grav_force(m1, m2, position_1, position_2):
    # Returns the force on mass 1
    return (
        (position_2 - position_1)
        * G
        * m1
        * m2
        / (np.linalg.norm(position_2 - position_1) ** 3)
    )

## Brute-Force Gravitational Force Computation

In [ ]:
def ds_dt(t, s):
    positions = s.reshape([-1, 3])[:N]
    velocities = s.reshape([-1, 3])[N:]
    accelerations = np.zeros((N, 3))

    # We need to just take the force once. We can add the accelerations to the array
    for i in range(N):
        for j in range(N):
            if i < j:
                force_ij = compute_grav_force(
                    masses[i], masses[j], positions[i], positions[j]
                )

                accelerations[i] += force_ij / masses[i]
                accelerations[j] += -force_ij / masses[j]
    return np.append(velocities, accelerations)

## Barnes-Hut Tree Force Computation
## TODO
- Need to figure out a way to store the centre of mass of the nodes in the class and how to update this iteratively
- Need to look at fast algorithms and how state-of-the-art optimises the algorithm
- Implement force calculation function from the tree
- Implement the loop

### Node Class

In [ ]:
class Node:
    def __init__(self, position, mass):
        # nw, ne, sw, se
        self.children = [None, None, None, None]
        self.parent = None
        self.position = position
        self.mass = mass

### Tree Class

In [ ]:
class Tree:
    def __init__(self):
        self.children = [None, None, None, None]

### Direction of particle

In [ ]:
def direction(origin, position):
    displacement = [position[0] - origin[0], position[1] - origin[1]]
    if displacment[0] >= 0 and displacement[1] >= 0:
        return 0
    if displacement[0] >= 0 and displacement[1] < 0:
        return 2
    if displacement[0] < 0 and displacement[1] >= 0:
        return 1
    if displacement[0] < 0 and displacement[1] < 0:
        return 3

### Add nodes to tree

In [ ]:
def add_node(positions, masses, parentNode):
    if positions.len() == 0:
        return None
    currentNode = Node(positions[0], masses[0])
    currentNode.parent = parentNode

### Construct Barnes-Hut tree

In [ ]:
def construct_barnes_hut_tree(positions, masses):
    tree = Tree()
    rootNode = Node(positions[0], masses[0])
    add_node(positions[1:], masses[1:], rootNode)

In [ ]:
# TODO
def ds_dt(t, s):

    raise NotImplementedError

## Estimate the period of orbits

In [ ]:
def period_estimation(solution):

    def centre_pass(t, s):
        x = s[6]
        return x

    centre_pass.direction = 1

    periods = []
    for i in range(len(solution.t_events[0]) - 1):
        periods.append(solution.t_events[0][i + 1] - solution.t_events[0][i])

    print(np.average(periods))
    print(np.std(periods))

## Graphics - Plot Trajectories

In [ ]:
def plot_trajectories(solution):
    ax = plt.figure().add_subplot(projection="3d")
    ax.set_aspect("equal")
    positions = []
    for i in range(N):
        positions.append(solution.y[i * 3 : i * 3 + 3])

    positions = np.array(positions)

    ax.set_xlim(np.min(positions), np.max(positions))
    ax.set_ylim(np.min(positions), np.max(positions))
    ax.set_zlim(np.min(positions), np.max(positions))

    for i in range(N):
        plt.plot(positions[i][0], positions[i][1], positions[i][2], label=labels[i])
    
    plt.legend()
    plt.show()

## Random Setup

### Parameters

In [ ]:
# 50 random masses.
N = 50

# Generate values of masses
masses = np.random.rand(N) * 1000

# Generate initial positions
positions = np.array([np.random.rand(3) for _ in range(N)]) * 10

# Generate initial velocities
velocities = np.array([np.random.rand(3) for _ in range(N)]) * 5

# Label each object
labels = [f"Object {i+1}" for i in range(N)]

# State will contain positions, then velocities
initial_state = np.append(positions, velocities)

### Execution

In [ ]:
solution = solve_ivp(
    ds_dt, [0, 10], initial_state, max_step=0.005
)  # events = centre_pass
t_values = solution.t

plot_trajectories(solution)

## Sun-Earth-Moon

### Parameters

In [ ]:
# Sun, Earth and Moon
N = 3
masses = np.array([10000, 100, 0.001])
positions = np.array([[0, 0, 0], [0, 0, 1], [0, 0, 1.05]]) * 10
earth_vel = (G * masses[0] / positions[1][2]) ** 0.5
moon_vel = (G * masses[1] / (positions[2][2] - positions[1][2])) ** 0.5
velocities = np.array(
    [
        [0, -earth_vel * masses[1] / masses[0], 0],
        [0, earth_vel, 0],
        [0, moon_vel * 1.5, 0],
    ]
)

labels = ["Sun", "Earth", "Moon"]

### Execution

In [ ]:
solution = solve_ivp(
    ds_dt, [0, 10], initial_state, max_step=0.005
)  # events = centre_pass
t_values = solution.t

print(period_estimation(solution))

plot_trajectories(solution)

## Galaxy

### Parameters

In [2]:
# TODO

### Execution

In [ ]:
# TODO